In [ ]:
# =============================================================================
# SECTION 1: IMPORTS AND SETUP
# =============================================================================

from __future__ import annotations
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Sequence, Tuple, Set
from pathlib import Path
from collections import Counter
import hashlib
import re
import logging
import traceback
import json
import pandas as pd

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

%cd /media/mohamed-gaber/01D96E17175F8760/Artifitial_intelligence/GraduationProject

print("✓ Imports complete")

# %%
# =============================================================================
# SECTION 2: HELPER FUNCTIONS
# =============================================================================

def _stable_id(s: str) -> str:
    """Generate a stable hash ID for a string."""
    return hashlib.sha256(s.encode("utf-8")).hexdigest()[:16]


def _dedup_rules(rules: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Remove duplicate rules while preserving order."""
    seen = set()
    out = []
    for r in rules:
        key = (r.get("type"), str(r.get("article_no")), r.get("text"))
        if key not in seen:
            seen.add(key)
            out.append(r)
    return out

print("✓ Helper functions defined")

# %%
# =============================================================================
# SECTION 3: DATA STRUCTURES
# =============================================================================

@dataclass
class ExtractedLaw:
    """Container for all extracted law components."""
    law_meta: Dict[str, Any]
    articles: List[Dict[str, Any]]          # [{article_no, text}]
    references: List[Dict[str, Any]]        # [{from_article_no, to_article_no}]

print("✓ Data structures defined")

/media/mohamed-gaber/01D96E17175F8760/Artifitial_intelligence/GraduationProject
✓ Imports complete
✓ Helper functions defined
✓ Data structures defined


In [21]:
# =============================================================================
# SECTION 4: BASE EXTRACTOR
# =============================================================================

class BaseLawExtractor:

    # Enhanced article marker pattern
    ARTICLE_MARK_RE = re.compile(
        r"""(?m)^\s*
            (?:المادة|مادة)\s*              # article keyword (required)
            (?:\(\s*)?                      # optional opening paren
            (?P<num>
                [0-9٠-٩]+                   # base number
                (?:
                    \s*
                    (?:
                        مكرر(?:[اأإآى])?    # مكرر with optional suffix
                        (?:\s*(?:[اأإآA-Za-z]))?
                        |
                        (?:أولا?ً?|ثانيا?ً?|ثالثا?ً?|رابعا?ً?|خامسا?ً?)
                    )
                )?
                (?:\s*\(\s*[اأإآA-Za-z]\s*\))?
            )
            (?:\s*\))?                      # optional closing paren
            \s*[:：\-–\s]                   # separator
            (?P<rest>.*)
            $
        """,
        re.VERBOSE
    )

    # Enhanced reference pattern
    REF_RE = re.compile(
        r"""
        (?:المادة|مادة|المواد)\s*
        (?:\(\s*)?
        (?P<num>
            [0-9٠-٩]+
            (?:\s*(?:مكرر(?:[اأإآى])?)?)?
            (?:\s*\(\s*[اأإآA-Za-z]\s*\))?
        )
        (?:\s*\))?
        (?:
            \s*(?:و|،|,)\s*
            (?:\(\s*)?
            (?P<num2>[0-9٠-٩]+(?:\s*مكرر(?:[اأإآى])?)?)
            (?:\s*\))?
        )?
        """,
        re.VERBOSE
    )


    _ARABIC_INDIC_TRANS = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")

    def __init__(self, raw_text: str, law_id: str, law_name: str):
        self.raw_text = raw_text or ""
        self._law_id = law_id
        self._law_name = law_name
        self._validation_warnings: List[str] = []

    def _to_western_digits(self, s: str) -> str:
        """Convert Arabic-Indic numerals to Western."""
        return s.translate(self._ARABIC_INDIC_TRANS)

    def _normalize_article_no(self, token: str) -> str:
        """Normalize article number to standard format."""
        token = token.strip()
        token = self._to_western_digits(token)
        token = re.sub(r"\s+", " ", token)
        token = re.sub(r"مكر(?:ر[اأإآىًٍَُِّْ]?)", "مكرر", token)
        
        ordinal_map = {
            "أولاً": "أولا", "أولا": "أولا", "أول": "أولا",
            "ثانياً": "ثانيا", "ثانيا": "ثانيا", "ثاني": "ثانيا",
            "ثالثاً": "ثالثا", "ثالثا": "ثالثا", "ثالث": "ثالثا",
            "رابعاً": "رابعا", "رابعا": "رابعا", "رابع": "رابعا",
        }
        for old, new in ordinal_map.items():
            token = token.replace(old, new)
        
        return token.strip()

    def extract_law_meta(self) -> Dict[str, Any]:
        """Extract law metadata."""
        return {
            "law_id": self._law_id,
            "name": self._law_name,
            "validation_warnings": self._validation_warnings
        }

    def extract_articles(self) -> List[Dict[str, str]]:
        """Extract articles with improved accuracy."""
        text = self.raw_text
        matches = list(self.ARTICLE_MARK_RE.finditer(text))

        articles: List[Dict[str, str]] = []

        if not matches:
            logger.warning("No article markers found in text")
            self._validation_warnings.append("No article markers found")
            return [{
                "article_no": None,
                "text": text.strip()
            }]

        logger.info(f"Found {len(matches)} article markers")

        # Extract preamble
        first = matches[0]
        if first.start() > 0:
            preamble = text[:first.start()].strip()
            if preamble:
                articles.append({
                    "article_no": None,
                    "text": preamble
                })
                logger.debug(f"Extracted preamble ({len(preamble)} chars)")

        # Extract each article
        for i, m in enumerate(matches):
            start = m.end()
            end = matches[i + 1].start() if i + 1 < len(matches) else len(text)

            body = text[start:end].strip()
            raw_no = m.group("num") or ""
            article_no = self._normalize_article_no(raw_no)

            if len(body) < 10 and i < len(matches) - 1:
                logger.warning(f"Article {article_no} is very short ({len(body)} chars)")
                self._validation_warnings.append(f"Short article: {article_no}")

            articles.append({
                "article_no": article_no,
                "text": body,
                "raw_article_no": raw_no
            })
            
            logger.debug(f"Extracted article {article_no} ({len(body)} chars)")

        logger.info(f"Total extracted: {len(articles)} articles")
        return articles

    def extract_references(self, articles: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """Extract internal article references."""
        refs = set()
        
        for a in articles:
            from_no = str(a.get("article_no", "")).strip()
            if not from_no or from_no == "None":
                continue
                
            text = a.get("text", "")
            
            for match in self.REF_RE.finditer(text):
                to_no = match.group("num")
                if to_no:
                    to_no = self._normalize_article_no(to_no)
                    if to_no and to_no != from_no:
                        refs.add((from_no, to_no))
                
                to_no2 = match.group("num2") if match.lastindex >= 2 else None
                if to_no2:
                    to_no2 = self._normalize_article_no(to_no2)
                    if to_no2 and to_no2 != from_no:
                        refs.add((from_no, to_no2))
        
        result = [{"from_article_no": f, "to_article_no": t} for f, t in sorted(refs)]
        logger.info(f"Extracted {len(result)} references")
        return result


    def extract_all(self) -> ExtractedLaw:
        """Extract all components."""
        logger.info(f"Starting extraction for {self._law_name}")
        
        meta = self.extract_law_meta()
        articles = self.extract_articles()
        references = self.extract_references(articles)

        logger.info(f"Extraction complete: {len(articles)} articles")

        return ExtractedLaw(
            law_meta=meta,
            articles=articles,
            references=references,
        )

print("✓ Base extractor defined")

✓ Base extractor defined


In [22]:
# =============================================================================
# SECTION 5: MIXINS (REUSABLE COMPONENTS)
# =============================================================================

class PenalRulesMixin:
    """Enhanced penal rule extractor."""

    _AR_DIACRITICS_RE = re.compile(r"[\u0617-\u061A\u064B-\u0652\u0670]")
    _TATWEEL_RE = re.compile(r"\u0640")

    @staticmethod
    def _normalize(text: str) -> str:
        """Normalize Arabic text for pattern matching."""
        if not text:
            return ""
        text = PenalRulesMixin._AR_DIACRITICS_RE.sub("", text)
        text = PenalRulesMixin._TATWEEL_RE.sub("", text)
        replacements = [
            ("أ", "ا"), ("إ", "ا"), ("آ", "ا"),
            ("ى", "ي"), ("ئ", "ي"),
            ("ة", "ه"), ("ؤ", "و")
        ]
        for old, new in replacements:
            text = text.replace(old, new)
        text = re.sub(r"\s+", " ", text).strip()
        return text


    @staticmethod
    def _split_into_chunks(text: str) -> List[str]:
        """Split text into logical chunks."""
        chunks = [p.strip() for p in text.split('\n\n') if p.strip()]
        result = []
        for chunk in chunks:
            if len(chunk) > 500:
                result.extend([line.strip() for line in chunk.split('\n') if line.strip()])
            else:
                result.append(chunk)
        return result if result else [text]


class ProcedureRulesMixin:
    """Enhanced procedure rule extractor."""

    _AR_DIACRITICS_RE = re.compile(r"[\u0617-\u061A\u064B-\u0652\u0670]")
    _TATWEEL_RE = re.compile(r"\u0640")

    @staticmethod
    def _normalize_ar(text: str) -> str:
        """Normalize Arabic text."""
        if not text:
            return ""
        text = ProcedureRulesMixin._AR_DIACRITICS_RE.sub("", text)
        text = ProcedureRulesMixin._TATWEEL_RE.sub("", text)
        replacements = [
            ("أ", "ا"), ("إ", "ا"), ("آ", "ا"),
            ("ى", "ي"), ("ئ", "ي"),
            ("ؤ", "و"), ("ة", "ه")
        ]
        for old, new in replacements:
            text = text.replace(old, new)
        text = re.sub(r"\s+", " ", text).strip()
        return text

    PROC_LINE_RE = re.compile(
        r"""(?ix).*?\b(?:يجب(?:\s+على?|\s+أن)?|يتعيّ?ن(?:\s+على?|\s+أن)?|يلزم(?:\s+على?|\s+أن)?|يلتزم(?:\s+ب)?|على\s+[\w\s]{1,60}?\s+أن|يتحتّ?م|يترتّ?ب\s+على?|يجوز(?:\s+ل)?|يُ?سمح(?:\s+ب)?|يُ?مكن|يحقّ?(?:\s+ل)?|لا\s+يجوز|لا\s+يُ?سمح|لا\s+يُ?مكن|يُ?حظر|يُ?منع|محظور|ممنوع|يشترط|بشرط\s+أن|شريطه?\s+أن|خاضع(?:\s+ل)?|مرهون(?:\s+ب)?|موقوف(?:\s+على)?)\b.*""",
        re.VERBOSE
    )

    PROC_LINE_EN_RE = re.compile(
        r"""(?ix).*?\b(?:must|shall|should|may\s+(?:not|only)?|is\s+(?:required|mandatory|obligatory|prohibited|forbidden|permitted)|shall\s+(?:not|be)|required\s+(?:to|that)|subject\s+to|conditional\s+(?:on|upon)|prohibited|forbidden)\b.*""",
        re.VERBOSE
    )

    @staticmethod
    def _split_text_chunks(text: str) -> List[str]:
        """Split text into chunks."""
        chunks = []
        buffer = []
        
        for line in text.splitlines():
            line = line.strip()
            if not line:
                if buffer:
                    chunks.append(" ".join(buffer))
                    buffer = []
                continue
            
            if len(line) > 200:
                sentences = re.split(r'[.。]\s+', line)
                for sent in sentences:
                    sent = sent.strip()
                    if sent:
                        if len(sent) > 300:
                            buffer.append(sent)
                            chunks.append(" ".join(buffer))
                            buffer = []
                        else:
                            buffer.append(sent)
            else:
                buffer.append(line)
        
        if buffer:
            chunks.append(" ".join(buffer))
        
        return chunks if chunks else [text]

In [23]:
# =============================================================================
# SECTION 6: LAW-SPECIFIC EXTRACTORS
# =============================================================================

class PenalCodeExtractor(BaseLawExtractor, PenalRulesMixin):
    """Penal Code extractor."""

    def __init__(self, raw_text: str):
        if not isinstance(raw_text, str):
            raise TypeError("raw_text must be a str")
        super().__init__(raw_text, law_id="penal_code", law_name="قانون العقوبات")



class MoneyLaunderingExtractor(BaseLawExtractor, PenalRulesMixin):
    """Money Laundering Law extractor."""

    def __init__(self, raw_text: str):
        if not isinstance(raw_text, str):
            raise TypeError("raw_text must be a str")
        super().__init__(raw_text, law_id="money_laundering", law_name="قانون مكافحة غسل الأموال")



class WeaponsAmmunitionExtractor(BaseLawExtractor, PenalRulesMixin):
    """Weapons and Ammunition Law extractor."""

    def __init__(self, raw_text: str):
        if not isinstance(raw_text, str):
            raise TypeError("raw_text must be a str")
        super().__init__(raw_text, law_id="weapons_ammunition", law_name="قانون الأسلحة والذخيرة")


class CriminalConstitutionExtractor(BaseLawExtractor):
    """Criminal Constitution extractor."""

    def __init__(self, raw_text: str):
        if not isinstance(raw_text, str):
            raise TypeError("raw_text must be a str")
        super().__init__(raw_text, law_id="criminal_constitution", law_name="الدستور الجنائي")



class DrugsLawExtractor(BaseLawExtractor, PenalRulesMixin):
    """Anti-Drugs Law extractor."""

    def __init__(self, raw_text: str):
        if not isinstance(raw_text, str):
            raise TypeError("raw_text must be a str")
        super().__init__(raw_text, law_id="anti_drugs", law_name="قانون مكافحة المخدرات")



class CriminalProcedureExtractor(BaseLawExtractor, ProcedureRulesMixin):
    """Criminal Procedure Law extractor."""

    def __init__(self, raw_text: str):
        if not isinstance(raw_text, str):
            raise TypeError("raw_text must be a str")
        super().__init__(raw_text, law_id="criminal_procedure", law_name="قانون الإجراءات الجنائية")


class AntiTerrorExtractor(BaseLawExtractor, PenalRulesMixin):
    """Anti-Terror Law extractor."""

    def __init__(self, raw_text: str):
        if not isinstance(raw_text, str):
            raise TypeError("raw_text must be a str")
        super().__init__(raw_text, law_id="anti_terror", law_name="قانون مكافحة الإرهاب")


class EmergencyLawExtractor(BaseLawExtractor):
    """Emergency Law extractor."""

    MEASURE_LINE_RE = re.compile(
        r"""(?ix)\b(?:يجوز(?:\s+ل)?|للرئيس|للسلط(?:ة|ات)|للجهات|تُ?منح\s+(?:السلطات|الصلاحيات)|يُ?خوّ?ل)\b.+""",
        re.VERBOSE
    )

    def __init__(self, raw_text: str):
        if not isinstance(raw_text, str):
            raise TypeError("raw_text must be a str")
        super().__init__(raw_text, law_id="emergency_law", law_name="قانون الطوارئ")


class CybercrimeExtractor(BaseLawExtractor, PenalRulesMixin, ProcedureRulesMixin):
    """Cybercrime Law extractor."""

    def __init__(self, raw_text: str):
        if not isinstance(raw_text, str):
            raise TypeError("raw_text must be a str")
        super().__init__(raw_text, law_id="cybercrime", law_name="قانون مكافحة جرائم تقنية المعلومات")


def get_extractor(law_key: str, raw_text: str) -> BaseLawExtractor:
    """Factory function to get the appropriate extractor."""
    mapping = {
        "penal_code": PenalCodeExtractor,
        "money_laundering": MoneyLaunderingExtractor,
        "weapons_ammunition": WeaponsAmmunitionExtractor,
        "criminal_constitution": CriminalConstitutionExtractor,
        "anti_drugs": DrugsLawExtractor,
        "criminal_procedure": CriminalProcedureExtractor,
        "anti_terror": AntiTerrorExtractor,
        "emergency_law": EmergencyLawExtractor,
        "cybercrime": CybercrimeExtractor,
    }
    
    if law_key not in mapping:
        raise KeyError(
            f"Unknown law_key: '{law_key}'. "
            f"Available keys: {sorted(mapping.keys())}"
        )
    
    cls = mapping[law_key]
    return cls(raw_text)

print("✓ Law extractors defined")

✓ Law extractors defined


In [24]:
# =============================================================================
# SECTION 7: FILE READING AND INGESTION
# =============================================================================

def _safe_read_text_file(
    fp: Path,
    encodings: Sequence[str],
    max_size_mb: Optional[int] = 50,
) -> Optional[str]:
    """Try reading a text file using a list of encodings."""
    try:
        size_mb = fp.stat().st_size / (1024 * 1024)
    except Exception:
        size_mb = 0.0

    if max_size_mb is not None and size_mb > max_size_mb:
        logger.warning("Skipping large file %s (%.1f MB > %d MB)", fp, size_mb, max_size_mb)
        return None

    for enc in encodings:
        try:
            content = fp.read_text(encoding=enc)
            logger.debug("Successfully read %s with encoding %s", fp.name, enc)
            return content
        except UnicodeDecodeError:
            try:
                content = fp.read_text(encoding=enc, errors="replace")
                logger.warning("Read %s with encoding %s (with replacements)", fp.name, enc)
                return content
            except Exception:
                continue
        except Exception as e:
            logger.debug("Failed reading %s with encoding %s: %s", fp, enc, e)
            continue

    try:
        raw = fp.read_bytes()
        content = raw.decode("utf-8", errors="replace")
        logger.warning("Read %s with UTF-8 fallback (may have errors)", fp.name)
        return content
    except Exception as e:
        logger.error("Failed to read file %s: %s", fp, e)
        return None


def read_all_txt_in_folder(
    folder: Path,
    encoding_list: Optional[Sequence[str]] = None,
    recursive: bool = False,
    file_pattern: str = "*.txt",
    max_file_size_mb: Optional[int] = 50,
) -> str:
    """Read and concatenate all .txt files in a folder."""
    folder = Path(folder)
    if not folder.exists() or not folder.is_dir():
        raise FileNotFoundError(f"Folder not found or not a directory: {folder}")

    encodings = list(encoding_list) if encoding_list else [
        "utf-8", "utf-16", "cp1256", "cp1252", "iso-8859-6", "latin1"
    ]

    globber = folder.rglob if recursive else folder.glob
    txt_files = sorted([p for p in globber(file_pattern) if p.is_file()])

    if not txt_files:
        logger.warning("No files matching '%s' found in %s", file_pattern, folder)
        return ""

    logger.info("Found %d file(s) matching '%s' in %s", len(txt_files), file_pattern, folder)

    parts: List[str] = []
    failed_files = []
    
    for fp in txt_files:
        try:
            text = _safe_read_text_file(fp, encodings=encodings, max_size_mb=max_file_size_mb)
            if text:
                parts.append(text)
                logger.debug("Read %s (%d chars)", fp.name, len(text))
            else:
                logger.warning("No content extracted from %s", fp.name)
                failed_files.append(fp.name)
        except Exception as e:
            logger.exception("Error reading file %s: %s", fp, e)
            failed_files.append(fp.name)

    if failed_files:
        logger.warning("Failed to read %d file(s): %s", len(failed_files), ", ".join(failed_files))

    result = "\n\n".join(parts).strip()
    logger.info("Total text length: %d characters", len(result))
    return result


def ingest_dataset(
    root_dir: str,
    folder_to_law_key: Dict[str, str],
    *,
    recursive_read: bool = False,
    encoding_list: Optional[Sequence[str]] = None,
    max_file_size_mb: Optional[int] = 50,
    verbose: bool = True,
    validate: bool = True,
) -> List[Dict[str, object]]:
    """Ingest entire dataset and extract all laws."""
    root = Path(root_dir)
    if not root.exists():
        raise FileNotFoundError(f"Root dir not found: {root}")

    summaries: List[Dict[str, object]] = []
    folders = sorted([p for p in root.iterdir() if p.is_dir()])
    logger.info("Found %d folder(s) in %s", len(folders), root)

    for folder in folders:
        folder_name = folder.name

        if folder_name not in folder_to_law_key:
            logger.info("[SKIP] No mapping for folder: %s", folder_name)
            continue

        law_key = folder_to_law_key[folder_name]
        logger.info("=" * 70)
        logger.info("Processing: %s -> %s", folder_name, law_key)
        logger.info("=" * 70)

        try:
            raw_text = read_all_txt_in_folder(
                folder,
                encoding_list=encoding_list,
                recursive=recursive_read,
                max_file_size_mb=max_file_size_mb,
            )
        except Exception as e:
            logger.exception("Failed reading folder %s: %s", folder_name, e)
            summaries.append({
                "folder": folder_name,
                "law_key": law_key,
                "law_id": None,
                "articles": 0,
                "references": 0,
                "error": f"read_error: {e}",
                "warnings": []
            })
            continue

        if not raw_text:
            logger.warning("[SKIP] Empty folder: %s", folder_name)
            summaries.append({
                "folder": folder_name,
                "law_key": law_key,
                "law_id": None,
                "articles": 0,
                "references": 0,
                "error": "empty_folder",
                "warnings": []
            })
            continue

        try:
            extractor = get_extractor(law_key, raw_text)
        except Exception as e:
            logger.exception("Failed to create extractor for %s (%s): %s", folder_name, law_key, e)
            summaries.append({
                "folder": folder_name,
                "law_key": law_key,
                "law_id": None,
                "articles": 0,
                "references": 0,
                "error": f"extractor_init_error: {e}",
                "warnings": []
            })
            continue

        try:
            bundle = extractor.extract_all()
        except Exception as e:
            logger.error("Error extracting data for %s: %s", folder_name, e)
            logger.debug(traceback.format_exc())
            summaries.append({
                "folder": folder_name,
                "law_key": law_key,
                "law_id": None,
                "articles": 0,
                "references": 0,
                "error": f"extraction_error: {e}",
                "warnings": []
            })
            continue

        def safe_len(obj):
            try:
                return len(obj) if obj is not None else 0
            except Exception:
                return 0

        law_id = bundle.law_meta.get("law_id") if bundle.law_meta else None
        n_articles = safe_len(bundle.articles)
        n_refs = safe_len(bundle.references)
        warnings = bundle.law_meta.get("validation_warnings", []) if bundle.law_meta else []

        if validate:
            if n_articles == 0:
                warnings.append("No articles extracted")
            elif n_articles == 1 and bundle.articles[0].get("article_no") is None:
                warnings.append("Only preamble extracted, no numbered articles")
            
            if n_articles > 0:
                avg_len = sum(len(a.get("text", "")) for a in bundle.articles) / n_articles
                if avg_len < 50:
                    warnings.append(f"Suspiciously short articles (avg {avg_len:.0f} chars)")
                if avg_len > 5000:
                    warnings.append(f"Suspiciously long articles (avg {avg_len:.0f} chars)")

        summary = {
            "folder": folder_name,
            "law_key": law_key,
            "law_id": law_id,
            "articles": n_articles,
            "references": n_refs,
            "error": None,
            "warnings": warnings
        }
        summaries.append(summary)

        if verbose:
            logger.info(
                "[OK] Ingested %s -> law_id=%s articles=%d terms=%d refs=%d topics=%d rules=%d",
                folder_name, law_id, n_articles, n_refs
            )
            if warnings:
                logger.warning("Warnings for %s: %s", folder_name, "; ".join(warnings))

    logger.info("=" * 70)
    logger.info("INGESTION COMPLETE")
    logger.info("=" * 70)
    total = len(summaries)
    success = len([s for s in summaries if s["error"] is None])
    logger.info("Total folders: %d", total)
    logger.info("Successful: %d", success)
    logger.info("Failed: %d", total - success)

    return summaries

print("✓ Ingestion functions defined")

✓ Ingestion functions defined


In [25]:
# =============================================================================
# SECTION 8: VALIDATION UTILITIES
# =============================================================================

class ExtractionValidator:
    """Validates extraction quality and identifies issues."""
    
    def __init__(self, bundle, raw_text: str):
        self.bundle = bundle
        self.raw_text = raw_text
        self.issues = []
        self.warnings = []
        self.stats = {}
    
    def validate_all(self) -> Dict[str, Any]:
        """Run all validation checks."""
        self.validate_articles()
        self.validate_article_numbering()
        self.validate_references()
        self.calculate_statistics()
        
        return {
            "issues": self.issues,
            "warnings": self.warnings,
            "stats": self.stats,
            "overall_score": self.calculate_score()
        }
    
    def validate_article_numbering(self):
        """Validate article number sequence."""
        articles = self.bundle.articles
        numbered = [a for a in articles if a.get("article_no") and str(a.get("article_no")) != "None"]
        
        if not numbered:
            self.issues.append("No numbered articles found")
            return
        
        article_nos = [a.get("article_no") for a in numbered]
        duplicates = [no for no, count in Counter(article_nos).items() if count > 1]
        if duplicates:
            self.issues.append(f"Duplicate article numbers: {duplicates}")
        
        numeric_articles = []
        for no in article_nos:
            match = re.match(r'(\d+)', str(no))
            if match:
                numeric_articles.append(int(match.group(1)))
        
        if numeric_articles:
            numeric_articles.sort()
            min_num = numeric_articles[0]
            max_num = numeric_articles[-1]
            expected_count = max_num - min_num + 1
            actual_count = len(set(numeric_articles))
            
            if actual_count < expected_count:
                missing = set(range(min_num, max_num + 1)) - set(numeric_articles)
                if missing and len(missing) < 20:
                    self.warnings.append(f"Potential gaps in numbering: missing {sorted(missing)}")
            
            self.stats["article_range"] = f"{min_num}-{max_num}"
            self.stats["expected_articles"] = expected_count
            self.stats["actual_articles"] = len(numbered)
    
    def validate_references(self):
        """Validate article references."""
        references = self.bundle.references
        articles = self.bundle.articles
        
        if not references:
            self.warnings.append("No references extracted")
            return
        
        valid_nos = set(
            str(a.get("article_no")) 
            for a in articles 
            if a.get("article_no") and str(a.get("article_no")) != "None"
        )
        
        self_refs = [r for r in references if r.get("from_article_no") == r.get("to_article_no")]
        if self_refs:
            self.warnings.append(f"Found {len(self_refs)} self-references")
        
        invalid_refs = []
        for r in references:
            from_no = str(r.get("from_article_no", ""))
            to_no = str(r.get("to_article_no", ""))
            
            if from_no not in valid_nos and from_no != "None":
                invalid_refs.append(f"from:{from_no}")
            if to_no not in valid_nos:
                invalid_refs.append(f"to:{to_no}")
        
        if invalid_refs:
            self.warnings.append(f"Found {len(set(invalid_refs))} references to missing articles")
        
        self.stats["references_count"] = len(references)
    
    def calculate_statistics(self):
        """Calculate overall statistics."""
        articles = self.bundle.articles
        
        if articles:
            lengths = [len(a.get("text", "")) for a in articles if a.get("article_no")]
            if lengths:
                self.stats["avg_article_length"] = sum(lengths) / len(lengths)
                self.stats["min_article_length"] = min(lengths)
                self.stats["max_article_length"] = max(lengths)
        
        n_articles = len([a for a in articles if a.get("article_no")])
        if n_articles > 0:
            self.stats["refs_per_article"] = len(self.bundle.references) / n_articles
    
    def calculate_score(self) -> float:
        """Calculate overall quality score (0-100)."""
        score = 100.0
        score -= len(self.issues) * 10
        score -= len(self.warnings) * 2
        
        coverage = self.stats.get("coverage_percent", 0)
        if 85 <= coverage <= 100:
            score += 5
        
        if self.bundle.references:
            score += 2
        
        return max(0, min(100, score))
    
    def print_report(self):
        """Print a formatted validation report."""
        print("\n" + "=" * 70)
        print("EXTRACTION VALIDATION REPORT")
        print("=" * 70)
        
        print(f"\nBasic Statistics:")
        print(f"  Articles: {len(self.bundle.articles)}")
        print(f"  References: {len(self.bundle.references)}")
        
        if self.issues:
            print(f"\n❌ ISSUES ({len(self.issues)}):")
            for issue in self.issues:
                print(f"  • {issue}")
        else:
            print("\n✓ No critical issues found")
        
        score = self.calculate_score()
        print(f"\n📊 Overall Quality Score: {score:.1f}/100")
        if score >= 90:
            print("   Grade: Excellent ✨")
        elif score >= 75:
            print("   Grade: Good ✓")
        elif score >= 60:
            print("   Grade: Acceptable ⚠️")
        else:
            print("   Grade: Needs Improvement ❌")
        
        print("\n" + "=" * 70 + "\n")

print("✓ Validation utilities defined")

✓ Validation utilities defined


In [26]:
# =============================================================================
# SECTION 9: CONFIGURATION
# =============================================================================

# SET YOUR PATHS HERE
ROOT_DIR = "Datasets/Unstructured_Data"
OUTPUT_JSON = "extraction_summary.json"

# Mapping of folder names to law keys
FOLDER_TO_LAW_KEY = {
    "3okobat": "penal_code",
    "8asl_amwal": "money_laundering",
    "Asle7a": "weapons_ammunition",
    "dostor_gena2y": "criminal_constitution",
    "drugs": "anti_drugs",
    "egra2at_gena2ya": "criminal_procedure",
    "erhab": "anti_terror",
    "taware2": "emergency_law",
    "technology": "cybercrime",
}

print("✓ Configuration set")
print(f"   Root directory: {ROOT_DIR}")
print(f"   Laws to extract: {len(FOLDER_TO_LAW_KEY)}")

✓ Configuration set
   Root directory: Datasets/Unstructured_Data
   Laws to extract: 9


In [27]:
# =============================================================================
# SECTION 10: RUN FULL EXTRACTION
# =============================================================================

print("\n" + "=" * 70)
print("STARTING FULL DATASET EXTRACTION")
print("=" * 70 + "\n")

try:
    summaries = ingest_dataset(
        root_dir=ROOT_DIR,
        folder_to_law_key=FOLDER_TO_LAW_KEY,
        recursive_read=False,
        max_file_size_mb=50,
        verbose=True,
        validate=True
    )
    
    print(f"\n✅ Successfully processed {len(summaries)} folders\n")
    
except Exception as e:
    print(f"\n❌ Error during ingestion: {e}\n")
    logger.exception("Full error details:")
    summaries = []

2026-02-15 02:04:48,130 - __main__ - INFO - Found 10 folder(s) in Datasets/Unstructured_Data
2026-02-15 02:04:48,130 - __main__ - INFO - ======================================================================
2026-02-15 02:04:48,130 - __main__ - INFO - Processing: 3okobat -> penal_code
2026-02-15 02:04:48,131 - __main__ - INFO - ======================================================================
2026-02-15 02:04:48,131 - __main__ - INFO - Found 3 file(s) matching '*.txt' in Datasets/Unstructured_Data/3okobat
2026-02-15 02:04:48,132 - __main__ - INFO - Total text length: 169853 characters
2026-02-15 02:04:48,132 - __main__ - INFO - Starting extraction for قانون العقوبات
2026-02-15 02:04:48,134 - __main__ - INFO - Found 540 article markers
2026-02-15 02:04:48,134 - __main__ - WARNING - Article 64 is very short (5 chars)
2026-02-15 02:04:48,135 - __main__ - WARNING - Article 65 is very short (5 chars)
2026-02-15 02:04:48,135 - __main__ - WARNING - Article 66 is very short (5 chars)
2026


STARTING FULL DATASET EXTRACTION



2026-02-15 02:04:48,259 - __main__ - WARNING - Article 207 is very short (0 chars)
2026-02-15 02:04:48,259 - __main__ - WARNING - Article 208 is very short (0 chars)
2026-02-15 02:04:48,260 - __main__ - INFO - Total extracted: 511 articles
2026-02-15 02:04:48,263 - __main__ - INFO - Extracted 86 references
2026-02-15 02:04:48,263 - __main__ - INFO - Extraction complete: 511 articles
--- Logging error ---
Traceback (most recent call last):
  File "/home/mohamed-gaber/.local/share/uv/python/cpython-3.13.0-linux-x86_64-gnu/lib/python3.13/logging/__init__.py", line 1150, in emit
    msg = self.format(record)
  File "/home/mohamed-gaber/.local/share/uv/python/cpython-3.13.0-linux-x86_64-gnu/lib/python3.13/logging/__init__.py", line 998, in format
    return fmt.format(record)
           ~~~~~~~~~~^^^^^^^^
  File "/home/mohamed-gaber/.local/share/uv/python/cpython-3.13.0-linux-x86_64-gnu/lib/python3.13/logging/__init__.py", line 711, in format
    record.message = record.getMessage()
       


✅ Successfully processed 9 folders



In [28]:
# %%
# =============================================================================
# SECTION 11: DISPLAY RESULTS
# =============================================================================

if summaries:
    # Convert to DataFrame
    df = pd.DataFrame(summaries)
    
    print("\n" + "=" * 70)
    print("EXTRACTION SUMMARY")
    print("=" * 70 + "\n")
    display(df)
    
    # Save to JSON
    output_path = Path(OUTPUT_JSON)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(
        json.dumps(summaries, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )
    print(f"\n✓ Wrote JSON summary to {output_path}")
    
    # Statistics
    print("\n" + "=" * 70)
    print("STATISTICS")
    print("=" * 70)
    
    total = len(summaries)
    success = len([s for s in summaries if s["error"] is None])
    
    print(f"\nTotal folders: {total}")
    print(f"Successful: {success}")
    print(f"Failed: {total - success}")
    print(f"\nTotal extracted:")
    print(f"  Articles: {sum(s.get('articles', 0) for s in summaries)}")
    print(f"  References: {sum(s.get('references', 0) for s in summaries)}")
    
else:
    print("\n⚠️  No summaries to display.")


EXTRACTION SUMMARY



,folder,law_key,law_id,articles,references,error,warnings
0,3okobat,penal_code,penal_code,541,104,None,"[Short article: 64, Short article: 65, Short a..."
1,8asl_amwal,money_laundering,money_laundering,21,12,None,[]
2,Asle7a,weapons_ammunition,weapons_ammunition,52,11,None,[]
3,dostor_gena2y,criminal_constitution,criminal_constitution,252,6,None,"[Short article: 2, Short article: 3, Short art..."
4,drugs,anti_drugs,anti_drugs,69,26,None,[]
5,egra2at_gena2ya,criminal_procedure,criminal_procedure,511,86,None,"[Short article: 19, Short article: 20, Short a..."
6,erhab,anti_terror,anti_terror,55,2,None,"[Short article: 4, Short article: 5, Short art..."
7,taware2,emergency_law,emergency_law,21,1,None,[]
8,technology,cybercrime,cybercrime,46,8,None,[]



✓ Wrote JSON summary to extraction_summary.json

STATISTICS

Total folders: 9
Successful: 9
Failed: 0

Total extracted:
  Articles: 1568
  References: 256


In [29]:
# =============================================================================
# SECTION 12: DETAILED VALIDATION FOR SPECIFIC LAW
# =============================================================================
# Choose a law to validate in detail
LAW_TO_VALIDATE = "penal_code"  # Change this to test different laws

print("\n" + "=" * 70)
print(f"DETAILED VALIDATION: {LAW_TO_VALIDATE}")
print("=" * 70 + "\n")

try:
    folder_names = [k for k, v in FOLDER_TO_LAW_KEY.items() if v == LAW_TO_VALIDATE]
    
    if not folder_names:
        print(f"❌ No folder mapped to law key: {LAW_TO_VALIDATE}")
    else:
        folder_name = folder_names[0]
        folder_path = Path(ROOT_DIR) / folder_name
        
        # Read raw text
        print(f"Reading from: {folder_path}")
        raw_text = read_all_txt_in_folder(folder_path)
        
        # Extract
        print(f"Extracting with: {LAW_TO_VALIDATE} extractor")
        extractor = get_extractor(LAW_TO_VALIDATE, raw_text)
        bundle = extractor.extract_all()
        
        # Show sample articles
        print("=" * 70)
        print("SAMPLE ARTICLES (first 10)")
        print("=" * 70 + "\n")
        
        for i, article in enumerate(bundle.articles[:10]):
            art_no = article.get("article_no", "N/A")
            text = article.get("text", "")
            
            print(f"[{i+1}] Article {art_no}")
            print(f"    Length: {len(text)} chars")
            print(f"    Preview: {text[:200]}...")
            print("-" * 70)
        
        # Show extracted components
        print("\n" + "=" * 70)
        print("EXTRACTED COMPONENTS SAMPLES")
        print("=" * 70)
        
        if bundle.references:
            print("\n🔗 REFERENCES (first 10):")
            for i, ref in enumerate(bundle.references[:10]):
                print(f"  [{i+1}] Article {ref.get('from_article_no')} → Article {ref.get('to_article_no')}")
        
except Exception as e:
    print(f"\n❌ Error during detailed validation: {e}")
    logger.exception("Full error details:")

2026-02-15 02:04:48,335 - __main__ - INFO - Found 3 file(s) matching '*.txt' in Datasets/Unstructured_Data/3okobat
2026-02-15 02:04:48,336 - __main__ - INFO - Total text length: 169853 characters
2026-02-15 02:04:48,336 - __main__ - INFO - Starting extraction for قانون العقوبات
2026-02-15 02:04:48,338 - __main__ - INFO - Found 540 article markers
2026-02-15 02:04:48,339 - __main__ - WARNING - Article 64 is very short (5 chars)
2026-02-15 02:04:48,339 - __main__ - WARNING - Article 65 is very short (5 chars)
2026-02-15 02:04:48,339 - __main__ - WARNING - Article 66 is very short (5 chars)
2026-02-15 02:04:48,339 - __main__ - WARNING - Article 67 is very short (6 chars)
2026-02-15 02:04:48,340 - __main__ - WARNING - Article 68 is very short (5 chars)
2026-02-15 02:04:48,340 - __main__ - WARNING - Article 69 is very short (5 chars)
2026-02-15 02:04:48,340 - __main__ - WARNING - Article 70 is very short (6 chars)
2026-02-15 02:04:48,340 - __main__ - WARNING - Article 71 is very short (5 ch


DETAILED VALIDATION: penal_code

Reading from: Datasets/Unstructured_Data/3okobat
Extracting with: penal_code extractor
SAMPLE ARTICLES (first 10)

[1] Article None
    Length: 282 chars
    Preview: **آخر تعديل: 15 أغسطس 2021 بالقانون 141 لسنة 2021 **
القانون رقم 58 لسنة 1937
___________________________________________
إصدار قانون العقوبات
نحن فاروق الأول ملك مصر
قرر مجلس الشيوخ ومجلس النواب القا...
----------------------------------------------------------------------
[2] Article 1
    Length: 152 chars
    Preview: يلغى قانون العقوبات الجاري العمل به أمام المحاكم الأهلية وقانون العقوبات الذي تطبقه المحاكم المختلطة ويستعاض عنهما بقانون العقوبات المرافق لهذا القانون....
----------------------------------------------------------------------
[3] Article 2
    Length: 209 chars
    Preview: على وزير الحقانية تنفيذ هذا القانون ويعمل به من 15 أكتوبر سنة 1937.
نأمر بأن يبصم هذا القانون بخاتم الدولة وأن ينشر في الجريدة الرسمية وينفذ كقانون من قوانين الدولة.
_________________________________

In [ ]:
# =============================================================================
# SECTION 13: EXPORT RESULTS
# =============================================================================

if 'bundle' in locals():
    output_dir = Path("extraction_output")
    output_dir.mkdir(exist_ok=True)
    
    print("\n" + "=" * 70)
    print("EXPORTING RESULTS")
    print("=" * 70 + "\n")
    
    # Save articles
    articles_df = pd.DataFrame(bundle.articles)
    articles_path = output_dir / f"{LAW_TO_VALIDATE}_articles.csv"
    articles_df.to_csv(articles_path, index=False, encoding='utf-8-sig')
    print(f"✓ Saved articles to {articles_path}")
    
    
    # Save references
    if bundle.references:
        refs_df = pd.DataFrame(bundle.references)
        refs_path = output_dir / f"{LAW_TO_VALIDATE}_references.csv"
        refs_df.to_csv(refs_path, index=False, encoding='utf-8-sig')
        print(f"✓ Saved references to {refs_path}")
    
    print(f"\n✓ All results saved to {output_dir}/\n")

# %%
print("\n" + "=" * 70)
print("✅ ALL PROCESSING COMPLETE!")
print("=" * 70)


EXPORTING RESULTS

✓ Saved articles to extraction_output/penal_code_articles.csv
✓ Saved references to extraction_output/penal_code_references.csv

✓ All results saved to extraction_output/


✅ ALL PROCESSING COMPLETE!

Next steps:
1. Review the extraction summary table above
2. Check quality scores (target: >90)
3. Review validation reports for warnings
4. Export and analyze specific laws
5. Adjust patterns if needed and re-run

For troubleshooting, check the validation reports and logs above.
For ~100% accuracy, iterate on laws with scores <90.



In [31]:
from neo4j import GraphDatabase
from typing import Dict, List, Any, Optional, Tuple
import logging
from datetime import datetime
import hashlib
import json
from src.Config.config import NEO4J_PASSWORD , NEO4J_USERNAME , NEO4J_URI
logger = logging.getLogger(__name__)

print("✓ Imports complete")

# =============================================================================
# KNOWLEDGE GRAPH SCHEMA
# =============================================================================

"""
GRAPH SCHEMA:

Nodes:
------
(:Law)
  - law_id: string (unique)
  - name: string
  - text_length: int
  - created_at: datetime
  - article_count: int

(:Article)
  - article_id: string (unique, hash of law_id + article_no)
  - article_no: string
  - text: string
  - text_length: int
  - law_id: string



Relationships:
-------------
(:Law)-[:CONTAINS]->(:Article)
  properties: position (order in law)

(:Article)-[:REFERENCES]->(:Article)
  properties: reference_type (direct, indirect)


Indexes:
--------
- Law.law_id
- Article.article_id
- Article.article_no
"""

print("✓ Schema defined")

✓ Imports complete
✓ Schema defined


In [32]:
# =============================================================================
# KNOWLEDGE GRAPH BUILDER CLASS
# =============================================================================

class LegalKnowledgeGraph:
    """
    Builds and manages a knowledge graph for legal data using Neo4j.
    """
    
    def __init__(self, uri: str, user: str, password: str):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))
        logger.info(f"Connected to Neo4j at {uri}")
        
        # Verify connection
        try:
            self.driver.verify_connectivity()
            logger.info("✓ Neo4j connection verified")
        except Exception as e:
            logger.error(f"Failed to connect to Neo4j: {e}")
            raise
    
    def close(self):
        """Close database connection."""
        self.driver.close()
        logger.info("Neo4j connection closed")
    
    def __enter__(self):
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        self.close()
    
    # =========================================================================
    # SCHEMA SETUP
    # =========================================================================
    
    def setup_schema(self, drop_existing: bool = False):
        """
        Set up database schema: constraints and indexes.
        
        Args:
            drop_existing: If True, drop all existing data first
        """
        with self.driver.session() as session:
            if drop_existing:
                logger.warning("Dropping all existing data...")
                session.run("MATCH (n) DETACH DELETE n")
                logger.info("✓ All data cleared")
            
            # Drop existing constraints (in case of schema changes)
            logger.info("Setting up constraints and indexes...")
            
            constraints = [
                # Unique constraints
                "CREATE CONSTRAINT law_id_unique IF NOT EXISTS FOR (l:Law) REQUIRE l.law_id IS UNIQUE",
                "CREATE CONSTRAINT article_id_unique IF NOT EXISTS FOR (a:Article) REQUIRE a.article_id IS UNIQUE",
            ]
            
            for constraint in constraints:
                try:
                    session.run(constraint)
                    logger.debug(f"Created: {constraint[:50]}...")
                except Exception as e:
                    logger.debug(f"Constraint might already exist: {e}")
            
            # Additional indexes for performance
            indexes = [
                "CREATE INDEX article_no_index IF NOT EXISTS FOR (a:Article) ON (a.article_no)",
                "CREATE INDEX law_id_index IF NOT EXISTS FOR (a:Article) ON (a.law_id)",
            ]
            
            for index in indexes:
                try:
                    session.run(index)
                    logger.debug(f"Created: {index[:50]}...")
                except Exception as e:
                    logger.debug(f"Index might already exist: {e}")
            
            logger.info("✓ Schema setup complete")
    
    # =========================================================================
    # HELPER FUNCTIONS
    # =========================================================================
    
    @staticmethod
    def _generate_id(*components) -> str:
        """Generate a stable ID from components."""
        combined = "|".join(str(c) for c in components)
        return hashlib.sha256(combined.encode("utf-8")).hexdigest()[:16]
    
    # =========================================================================
    # NODE CREATION
    # =========================================================================
    
    def create_law(self, law_meta: Dict[str, Any]) -> str:
        """
        Create a Law node.
        
        Args:
            law_meta: Dictionary with law metadata
            
        Returns:
            law_id
        """
        with self.driver.session() as session:
            query = """
            MERGE (l:Law {law_id: $law_id})
            SET l.name = $name,
                l.text_length = $text_length,
                l.created_at = datetime(),
                l.updated_at = datetime()
            RETURN l.law_id as law_id
            """
            
            result = session.run(
                query,
                law_id=law_meta.get("law_id"),
                name=law_meta.get("name"),
                text_length=law_meta.get("text_length", 0)
            )
            
            record = result.single()
            law_id = record["law_id"] if record else law_meta.get("law_id")
            logger.info(f"✓ Created Law: {law_id}")
            return law_id
    
    def create_article(
        self, 
        law_id: str, 
        article_no: Optional[str], 
        text: str,
        position: int
    ) -> str:
        """
        Create an Article node and link it to a Law.
        
        Args:
            law_id: ID of the parent law
            article_no: Article number (can be None for preamble)
            text: Article text
            position: Position in the law (0-indexed)
            
        Returns:
            article_id
        """
        article_id = self._generate_id(law_id, article_no or "preamble", position)
        
        with self.driver.session() as session:
            query = """
            MATCH (l:Law {law_id: $law_id})
            MERGE (a:Article {article_id: $article_id})
            SET a.article_no = $article_no,
                a.text = $text,
                a.text_length = $text_length,
                a.law_id = $law_id,
                a.updated_at = datetime()
            MERGE (l)-[r:CONTAINS]->(a)
            SET r.position = $position
            RETURN a.article_id as article_id
            """
            
            result = session.run(
                query,
                law_id=law_id,
                article_id=article_id,
                article_no=article_no,
                text=text,
                text_length=len(text),
                position=position
            )
            
            record = result.single()
            logger.debug(f"✓ Created Article: {article_no} (position {position})")
            return article_id
    

    # =========================================================================
    # RELATIONSHIP CREATION
    # =========================================================================
    
    def create_article_reference(
        self,
        law_id: str,
        from_article_no: str,
        to_article_no: str
    ):
        """
        Create a REFERENCES relationship between articles.
        
        Args:
            law_id: ID of the law
            from_article_no: Source article number
            to_article_no: Target article number
        """
        with self.driver.session() as session:
            query = """
            MATCH (l:Law {law_id: $law_id})
            MATCH (l)-[:CONTAINS]->(from:Article {article_no: $from_article_no})
            MATCH (l)-[:CONTAINS]->(to:Article {article_no: $to_article_no})
            WHERE from.law_id = $law_id AND to.law_id = $law_id
            MERGE (from)-[r:REFERENCES]->(to)
            SET r.created_at = datetime()
            """
            
            session.run(
                query,
                law_id=law_id,
                from_article_no=from_article_no,
                to_article_no=to_article_no
            )
            
            logger.debug(f"✓ Created reference: {from_article_no} -> {to_article_no}")
    
    # =========================================================================
    # BATCH IMPORT FROM EXTRACTION
    # =========================================================================
    
    def import_law(self, bundle, verbose: bool = True) -> Dict[str, int]:
        """
        Import a complete law from extraction bundle.
        
        Args:
            bundle: ExtractedLaw object from extraction
            verbose: If True, print progress
            
        Returns:
            Dictionary with counts of imported items
        """
        stats = {
            "laws": 0,
            "articles": 0,
            "references": 0
        }
        
        law_id = bundle.law_meta.get("law_id")
        law_name = bundle.law_meta.get("name")
        
        if verbose:
            print(f"\n{'='*70}")
            print(f"IMPORTING: {law_name} ({law_id})")
            print(f"{'='*70}\n")
        
        # 1. Create Law node
        self.create_law(bundle.law_meta)
        stats["laws"] = 1
        if verbose:
            print(f"✓ Created law node")
        
        # 2. Create Article nodes
        article_map = {}  # Map article_no to article_id
        for i, article in enumerate(bundle.articles):
            article_no = article.get("article_no")
            text = article.get("text", "")
            
            article_id = self.create_article(law_id, article_no, text, i)
            if article_no:
                article_map[article_no] = article_id
            
            stats["articles"] += 1
        
        if verbose:
            print(f"✓ Created {stats['articles']} articles")
        
        # 6. Create Reference relationships
        for ref in bundle.references:
            self.create_article_reference(
                law_id=law_id,
                from_article_no=ref.get("from_article_no"),
                to_article_no=ref.get("to_article_no")
            )
            stats["references"] += 1
        
        if verbose:
            print(f"✓ Created {stats['references']} references")
        
        # Update law with article count
        with self.driver.session() as session:
            session.run(
                "MATCH (l:Law {law_id: $law_id}) SET l.article_count = $count",
                law_id=law_id,
                count=stats["articles"]
            )
        
        if verbose:
            print(f"\n{'='*70}")
            print(f"✅ IMPORT COMPLETE: {law_name}")
            print(f"{'='*70}\n")
        
        return stats
    
    def import_all_laws(
        self,
        summaries: List[Dict[str, Any]],
        extraction_function,
        verbose: bool = True
    ) -> Dict[str, Any]:
        """
        Import all laws from extraction summaries.
        
        Args:
            summaries: List of extraction summaries from ingest_dataset
            extraction_function: Function that takes (law_key, folder) and returns bundle
            verbose: If True, print progress
            
        Returns:
            Dictionary with total statistics
        """
        total_stats = {
            "laws": 0,
            "articles": 0,
            "references": 0,
            "errors": []
        }
        
        successful = [s for s in summaries if s.get("error") is None]
        
        if verbose:
            print(f"\n{'='*70}")
            print(f"IMPORTING {len(successful)} LAWS TO KNOWLEDGE GRAPH")
            print(f"{'='*70}\n")
        
        for summary in successful:
            law_key = summary.get("law_key")
            folder = summary.get("folder")
            
            try:
                # Get the bundle using extraction function
                bundle = extraction_function(law_key, folder)
                
                # Import to graph
                stats = self.import_law(bundle, verbose=verbose)
                
                # Accumulate stats
                for key in ["laws", "articles","references"]:
                    total_stats[key] += stats.get(key, 0)
                
            except Exception as e:
                logger.error(f"Failed to import {law_key}: {e}")
                total_stats["errors"].append({
                    "law_key": law_key,
                    "error": str(e)
                })
                if verbose:
                    print(f"❌ Error importing {law_key}: {e}\n")
        
        if verbose:
            print(f"\n{'='*70}")
            print(f"✅ ALL IMPORTS COMPLETE")
            print(f"{'='*70}")
            print(f"Total imported:")
            print(f"  Laws: {total_stats['laws']}")
            print(f"  Articles: {total_stats['articles']}")
            print(f"  References: {total_stats['references']}")
            if total_stats['errors']:
                print(f"  Errors: {len(total_stats['errors'])}")
            print(f"{'='*70}\n")
        
        return total_stats
    
    # =========================================================================
    # STATISTICS
    # =========================================================================
    
    def get_statistics(self) -> Dict[str, Any]:
        """
        Get comprehensive statistics about the knowledge graph.
        
        Returns:
            Dictionary with statistics
        """
        with self.driver.session() as session:
            stats = {}
            
            # Node counts
            stats["node_counts"] = {}
            for label in ["Law", "Article"]:
                result = session.run(f"MATCH (n:{label}) RETURN count(n) as count")
                stats["node_counts"][label] = result.single()["count"]
            
            # Relationship counts
            stats["relationship_counts"] = {}
            for rel_type in ["CONTAINS", "REFERENCES", "BELONGS_TO"]:
                result = session.run(f"MATCH ()-[r:{rel_type}]->() RETURN count(r) as count")
                stats["relationship_counts"][rel_type] = result.single()["count"]
            
            # Most referenced articles
            result = session.run("""
                MATCH (a:Article)<-[:REFERENCES]-(other)
                RETURN a.law_id as law_id, a.article_no as article_no, count(other) as ref_count
                ORDER BY ref_count DESC
                LIMIT 10
            """)
            stats["most_referenced_articles"] = [
                {
                    "law_id": r["law_id"],
                    "article_no": r["article_no"],
                    "reference_count": r["ref_count"]
                }
                for r in result
            ]
            
            return stats
    
    def print_statistics(self):
        """Print formatted statistics."""
        stats = self.get_statistics()
        
        print("\n" + "="*70)
        print("KNOWLEDGE GRAPH STATISTICS")
        print("="*70 + "\n")
        
        print("Node Counts:")
        for label, count in stats["node_counts"].items():
            print(f"  {label}: {count}")
        
        print("\nRelationship Counts:")
        for rel_type, count in stats["relationship_counts"].items():
            print(f"  {rel_type}: {count}")
        
        print("\nMost Referenced Articles:")
        for item in stats["most_referenced_articles"][:5]:
            print(f"  {item['law_id']} - Article {item['article_no']}: {item['reference_count']} references")
        
        print("\n" + "="*70 + "\n")

print("✓ Knowledge graph class defined")

✓ Knowledge graph class defined


In [33]:
# =============================================================================
# QUERY FUNCTIONS
# =============================================================================

class KnowledgeGraphQueries:
    """Useful queries for the legal knowledge graph."""
    
    def __init__(self, graph: LegalKnowledgeGraph):
        self.graph = graph
    
    def find_article(self, law_id: str, article_no: str) -> Dict[str, Any]:
        """Find a specific article and its connections."""
        with self.graph.driver.session() as session:
            query = """
            MATCH (l:Law {law_id: $law_id})-[:CONTAINS]->(a:Article {article_no: $article_no})
            OPTIONAL MATCH (a)-[:REFERENCES]->(ref:Article)
            RETURN a, 
                   collect(DISTINCT ref.article_no) as references
            """
            
            result = session.run(query, law_id=law_id, article_no=article_no)
            record = result.single()
            
            if not record:
                return None
            
            return {
                "article": dict(record["a"]),
                "references": record["references"]
            }
    
    def search_articles_by_text(self, search_term: str, limit: int = 10) -> List[Dict]:
        """Search articles containing specific text."""
        with self.graph.driver.session() as session:
            query = """
            MATCH (l:Law)-[:CONTAINS]->(a:Article)
            WHERE a.text CONTAINS $search_term
            RETURN l.name as law_name, a.article_no as article_no, 
                   a.text as text, a.text_length as text_length
            LIMIT $limit
            """
            
            result = session.run(query, search_term=search_term, limit=limit)
            return [dict(r) for r in result]
    
    def find_related_articles(
        self,
        law_id: str,
        article_no: str,
        max_depth: int = 2
    ) -> List[Dict]:
        """Find articles related through references (up to max_depth hops)."""
        with self.graph.driver.session() as session:
            query = f"""
            MATCH (l:Law {{law_id: $law_id}})-[:CONTAINS]->(start:Article {{article_no: $article_no}})
            MATCH path = (start)-[:REFERENCES*1..{max_depth}]->(related:Article)
            RETURN DISTINCT related.article_no as article_no, 
                   length(path) as distance,
                   related.text as text
            ORDER BY distance, article_no
            """
            
            result = session.run(query, law_id=law_id, article_no=article_no)
            return [dict(r) for r in result]

print("✓ Query functions defined")

# %%
print("\n" + "="*70)
print("✅ KNOWLEDGE GRAPH MODULE LOADED")
print("="*70)

✓ Query functions defined

✅ KNOWLEDGE GRAPH MODULE LOADED


In [34]:
# =============================================================================
# SECTION 1: IMPORTS
# =============================================================================

# Import extraction system (from complete_extraction_system.py)
import sys
from pathlib import Path
import logging
import pandas as pd
from typing import Dict, List, Any

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print("✓ All imports complete")

# %%
# =============================================================================
# SECTION 2: CONFIGURATION
# =============================================================================

# Extraction Configuration
ROOT_DIR = "Datasets/Unstructured_Data"
FOLDER_TO_LAW_KEY = {
    "3okobat": "penal_code",
    "8asl_amwal": "money_laundering",
    "Asle7a": "weapons_ammunition",
    "dostor_gena2y": "criminal_constitution",
    "drugs": "anti_drugs",
    "egra2at_gena2ya": "criminal_procedure",
    "erhab": "anti_terror",
    "taware2": "emergency_law",
    "technology": "cybercrime",
}

# Neo4j Configuration
NEO4J_URI = NEO4J_URI
NEO4J_USER = NEO4J_USERNAME
NEO4J_PASSWORD = NEO4J_PASSWORD

# Options
DROP_EXISTING_DATA = True  # Set to True to clear database before import
VALIDATE_EXTRACTION = True  # Set to True to validate each extraction

print("✓ Configuration set")
print(f"   Data directory: {ROOT_DIR}")
print(f"   Neo4j URI: {NEO4J_URI}")
print(f"   Neo4j User: {NEO4J_USER}")
print(f"   Laws to process: {len(FOLDER_TO_LAW_KEY)}")

✓ All imports complete
✓ Configuration set
   Data directory: Datasets/Unstructured_Data
   Neo4j URI: neo4j+s://8e3595a2.databases.neo4j.io
   Neo4j User: neo4j
   Laws to process: 9


In [35]:
# =============================================================================
# SECTION 3: EXTRACT ALL LAWS
# =============================================================================

print("\n" + "="*70)
print("STEP 1: EXTRACTING ALL LAWS")
print("="*70 + "\n")

summaries = ingest_dataset(
    root_dir=ROOT_DIR,
    folder_to_law_key=FOLDER_TO_LAW_KEY,
    recursive_read=False,
    max_file_size_mb=50,
    verbose=True,
    validate=VALIDATE_EXTRACTION
)

# Display summary
df = pd.DataFrame(summaries)
print("\n" + "="*70)
print("EXTRACTION SUMMARY")
print("="*70 + "\n")
display(df)

successful_summaries = [s for s in summaries if s.get("error") is None]
print(f"\n✓ Successfully extracted {len(successful_summaries)}/{len(summaries)} laws\n")

2026-02-15 02:04:48,414 - __main__ - INFO - Found 10 folder(s) in Datasets/Unstructured_Data
2026-02-15 02:04:48,414 - __main__ - INFO - ======================================================================
2026-02-15 02:04:48,415 - __main__ - INFO - Processing: 3okobat -> penal_code
2026-02-15 02:04:48,415 - __main__ - INFO - ======================================================================
2026-02-15 02:04:48,416 - __main__ - INFO - Found 3 file(s) matching '*.txt' in Datasets/Unstructured_Data/3okobat
2026-02-15 02:04:48,417 - __main__ - INFO - Total text length: 169853 characters
2026-02-15 02:04:48,417 - __main__ - INFO - Starting extraction for قانون العقوبات
2026-02-15 02:04:48,419 - __main__ - INFO - Found 540 article markers
2026-02-15 02:04:48,419 - __main__ - WARNING - Article 64 is very short (5 chars)
2026-02-15 02:04:48,420 - __main__ - WARNING - Article 65 is very short (5 chars)
2026-02-15 02:04:48,420 - __main__ - WARNING - Article 66 is very short (5 chars)
2026


STEP 1: EXTRACTING ALL LAWS



2026-02-15 02:04:48,504 - __main__ - INFO - Starting extraction for قانون الإجراءات الجنائية
2026-02-15 02:04:48,505 - __main__ - INFO - Found 510 article markers
2026-02-15 02:04:48,505 - __main__ - WARNING - Article 19 is very short (0 chars)
2026-02-15 02:04:48,505 - __main__ - WARNING - Article 20 is very short (0 chars)
2026-02-15 02:04:48,506 - __main__ - WARNING - Article 68 is very short (0 chars)
2026-02-15 02:04:48,506 - __main__ - WARNING - Article 135 is very short (0 chars)
2026-02-15 02:04:48,506 - __main__ - WARNING - Article 198 is very short (0 chars)
2026-02-15 02:04:48,507 - __main__ - WARNING - Article 207 is very short (0 chars)
2026-02-15 02:04:48,507 - __main__ - WARNING - Article 208 is very short (0 chars)
2026-02-15 02:04:48,508 - __main__ - INFO - Total extracted: 511 articles
2026-02-15 02:04:48,510 - __main__ - INFO - Extracted 86 references
2026-02-15 02:04:48,510 - __main__ - INFO - Extraction complete: 511 articles
--- Logging error ---
Traceback (most r


EXTRACTION SUMMARY



,folder,law_key,law_id,articles,references,error,warnings
0,3okobat,penal_code,penal_code,541,104,None,"[Short article: 64, Short article: 65, Short a..."
1,8asl_amwal,money_laundering,money_laundering,21,12,None,[]
2,Asle7a,weapons_ammunition,weapons_ammunition,52,11,None,[]
3,dostor_gena2y,criminal_constitution,criminal_constitution,252,6,None,"[Short article: 2, Short article: 3, Short art..."
4,drugs,anti_drugs,anti_drugs,69,26,None,[]
5,egra2at_gena2ya,criminal_procedure,criminal_procedure,511,86,None,"[Short article: 19, Short article: 20, Short a..."
6,erhab,anti_terror,anti_terror,55,2,None,"[Short article: 4, Short article: 5, Short art..."
7,taware2,emergency_law,emergency_law,21,1,None,[]
8,technology,cybercrime,cybercrime,46,8,None,[]



✓ Successfully extracted 9/9 laws



In [36]:
# =============================================================================
# SECTION 4: INITIALIZE KNOWLEDGE GRAPH
# =============================================================================

print("\n" + "="*70)
print("STEP 2: CONNECTING TO NEO4J")
print("="*70 + "\n")

try:
    graph = LegalKnowledgeGraph(
        uri=NEO4J_URI,
        user=NEO4J_USER,
        password=NEO4J_PASSWORD
    )
    
    # Setup schema
    graph.setup_schema(drop_existing=DROP_EXISTING_DATA)
    
    print("✓ Connected to Neo4j")
    print("✓ Schema setup complete")
    
except Exception as e:
    print(f"❌ Failed to connect to Neo4j: {e}")
    print("\nPlease ensure:")
    print("  1. Neo4j is running")
    print("  2. NEO4J_PASSWORD is correct")
    print("  3. Connection details are correct")
    raise

2026-02-15 02:04:48,544 - __main__ - INFO - Connected to Neo4j at neo4j+s://8e3595a2.databases.neo4j.io



STEP 2: CONNECTING TO NEO4J



2026-02-15 02:04:49,450 - __main__ - INFO - ✓ Neo4j connection verified
2026-02-15 02:04:49,450 - __main__ - WARNING - Dropping all existing data...
2026-02-15 02:04:49,551 - __main__ - INFO - ✓ All data cleared
2026-02-15 02:04:49,551 - __main__ - INFO - Setting up constraints and indexes...
2026-02-15 02:04:49,637 - neo4j.notifications - INFO - Received notification from DBMS server: <GqlStatusObject gql_status='00NA0', status_description="note: successful completion - index or constraint already exists. The command 'CREATE CONSTRAINT law_id_unique IF NOT EXISTS FOR (e:Law) REQUIRE (e.law_id) IS UNIQUE' has no effect. The index or constraint specified by 'CONSTRAINT constraint_b9269926 FOR (e:Law) REQUIRE (e.law_id) IS UNIQUE' already exists.", position=None, raw_classification='SCHEMA', classification=<NotificationClassification.SCHEMA: 'SCHEMA'>, raw_severity='INFORMATION', severity=<NotificationSeverity.INFORMATION: 'INFORMATION'>, diagnostic_record={'_classification': 'SCHEMA', '

✓ Connected to Neo4j
✓ Schema setup complete


In [38]:
# =============================================================================
# SECTION 5: IMPORT LAWS TO KNOWLEDGE GRAPH
# =============================================================================

print("\n" + "="*70)
print("STEP 3: BUILDING KNOWLEDGE GRAPH")
print("="*70 + "\n")

# Function to extract a specific law
def extract_law(law_key: str, folder_name: str) -> ExtractedLaw:
    """Extract a law given its key and folder name."""
    folder_path = Path(ROOT_DIR) / folder_name
    raw_text = read_all_txt_in_folder(folder_path)
    extractor = get_extractor(law_key, raw_text)
    bundle = extractor.extract_all()
    return bundle

# Import all laws
total_stats = {
    "laws": 0,
    "articles": 0,
    "references": 0,
    "errors": []
}

for summary in successful_summaries:
    law_key = summary.get("law_key")
    folder = summary.get("folder")
    
    try:
        print(f"\n{'='*70}")
        print(f"Importing: {law_key}")
        print(f"{'='*70}\n")
        
        # Extract
        bundle = extract_law(law_key, folder)
        
        # Import to graph
        stats = graph.import_law(bundle, verbose=True)
        
        # Accumulate stats
        for key in ["laws", "articles", "references"]:
            total_stats[key] += stats.get(key, 0)
        
    except Exception as e:
        logger.error(f"Failed to import {law_key}: {e}")
        total_stats["errors"].append({"law_key": law_key, "error": str(e)})
        print(f"❌ Error: {e}\n")

# Display total statistics
print("\n" + "="*70)
print("✅ KNOWLEDGE GRAPH BUILD COMPLETE")
print("="*70)
print(f"\nTotal imported:")
print(f"  Laws: {total_stats['laws']}")
print(f"  Articles: {total_stats['articles']}")
print(f"  References: {total_stats['references']}")

if total_stats['errors']:
    print(f"\n⚠️  Errors: {len(total_stats['errors'])}")
    for err in total_stats['errors']:
        print(f"    {err['law_key']}: {err['error']}")

print("\n" + "="*70 + "\n")

2026-02-15 02:05:22,067 - __main__ - INFO - Found 3 file(s) matching '*.txt' in Datasets/Unstructured_Data/3okobat
2026-02-15 02:05:22,068 - __main__ - INFO - Total text length: 169853 characters
2026-02-15 02:05:22,068 - __main__ - INFO - Starting extraction for قانون العقوبات
2026-02-15 02:05:22,070 - __main__ - INFO - Found 540 article markers
2026-02-15 02:05:22,070 - __main__ - WARNING - Article 64 is very short (5 chars)
2026-02-15 02:05:22,071 - __main__ - WARNING - Article 65 is very short (5 chars)
2026-02-15 02:05:22,071 - __main__ - WARNING - Article 66 is very short (5 chars)
2026-02-15 02:05:22,071 - __main__ - WARNING - Article 67 is very short (6 chars)
2026-02-15 02:05:22,072 - __main__ - WARNING - Article 68 is very short (5 chars)
2026-02-15 02:05:22,072 - __main__ - WARNING - Article 69 is very short (5 chars)
2026-02-15 02:05:22,072 - __main__ - WARNING - Article 70 is very short (6 chars)
2026-02-15 02:05:22,073 - __main__ - WARNING - Article 71 is very short (5 ch


STEP 3: BUILDING KNOWLEDGE GRAPH


Importing: penal_code


IMPORTING: قانون العقوبات (penal_code)



2026-02-15 02:05:22,856 - __main__ - INFO - ✓ Created Law: penal_code


✓ Created law node
✓ Created 541 articles


2026-02-15 02:06:29,267 - __main__ - INFO - Found 1 file(s) matching '*.txt' in Datasets/Unstructured_Data/8asl_amwal
2026-02-15 02:06:29,267 - __main__ - INFO - Total text length: 9819 characters
2026-02-15 02:06:29,268 - __main__ - INFO - Starting extraction for قانون مكافحة غسل الأموال
2026-02-15 02:06:29,268 - __main__ - INFO - Found 20 article markers
2026-02-15 02:06:29,268 - __main__ - INFO - Total extracted: 21 articles
2026-02-15 02:06:29,269 - __main__ - INFO - Extracted 12 references
2026-02-15 02:06:29,269 - __main__ - INFO - Extraction complete: 21 articles


...

✓ Created 2 references

✅ IMPORT COMPLETE: قانون مكافحة الإرهاب


Importing: emergency_law


IMPORTING: قانون الطوارئ (emergency_law)



2026-02-15 02:08:43,231 - __main__ - INFO - ✓ Created Law: emergency_law


✓ Created law node
✓ Created 21 articles
✓ Created 1 references


2026-02-15 02:08:46,765 - __main__ - INFO - Found 1 file(s) matching '*.txt' in Datasets/Unstructured_Data/technology
2026-02-15 02:08:46,765 - __main__ - INFO - Total text length: 30544 characters
2026-02-15 02:08:46,765 - __main__ - INFO - Starting extraction for قانون مكافحة جرائم تقنية المعلومات
2026-02-15 02:08:46,766 - __main__ - INFO - Found 45 article markers
2026-02-15 02:08:46,766 - __main__ - INFO - Total extracted: 46 articles
2026-02-15 02:08:46,766 - __main__ - INFO - Extracted 8 references
2026-02-15 02:08:46,767 - __main__ - INFO - Extraction complete: 46 articles
2026-02-15 02:08:46,900 - __main__ - INFO - ✓ Created Law: cybercrime



✅ IMPORT COMPLETE: قانون الطوارئ


Importing: cybercrime


IMPORTING: قانون مكافحة جرائم تقنية المعلومات (cybercrime)

✓ Created law node
✓ Created 46 articles
✓ Created 8 references

✅ IMPORT COMPLETE: قانون مكافحة جرائم تقنية المعلومات


✅ KNOWLEDGE GRAPH BUILD COMPLETE

Total imported:
  Laws: 9
  Articles: 1568
  References: 256




In [39]:
# =============================================================================
# SECTION 6: KNOWLEDGE GRAPH STATISTICS
# =============================================================================

print("\n" + "="*70)
print("STEP 4: ANALYZING KNOWLEDGE GRAPH")
print("="*70 + "\n")

graph.print_statistics()


STEP 4: ANALYZING KNOWLEDGE GRAPH


KNOWLEDGE GRAPH STATISTICS

Node Counts:
  Law: 9
  Article: 1568

Relationship Counts:
  CONTAINS: 1568
  REFERENCES: 446
  BELONGS_TO: 0

Most Referenced Articles:
  penal_code - Article 77: 24 references
  penal_code - Article 17: 18 references
  anti_drugs - Article 37: 18 references
  penal_code - Article 87: 17 references
  penal_code - Article 82: 15 references


